# Agente de Soporte al Cliente con PydanticAI + Ollama local

Version 2 de `pydanticai-101.ipynb`, adaptada para correr 100% en local con **Ollama**
(sin OpenAI API key, sin salida de datos del cliente a un tercero — soberania del dato
y soberania de la IA).

Cambios respecto al original:
- El `Agent` usa el endpoint OpenAI-compatible de Ollama (`http://localhost:11434/v1`),
  no la API de OpenAI.
- Modelo y parametros leidos de `config.ini` (no hardcodeados).
- CSV leido desde archivo local (`customer_support_tickets.csv`), no desde una URL externa.
- Se corrige `search_tickets` (en el notebook original la funcion se usaba pero nunca se
  definia, y el bloque de `prompt` tenia una indentacion invalida que rompia la celda).
- Se agrega una celda de prueba no interactiva (sin `input()`) para poder validar el
  agente durante desarrollo/CI, ademas del chat interactivo original.

**Requisito previo**: tener Ollama corriendo (`ollama serve`, normalmente ya activo como
servicio) y el modelo descargado (`ollama pull qwen2.5:7b-instruct-q4_K_M`).


In [ ]:
import configparser
from pathlib import Path

config = configparser.ConfigParser()
config.read(Path.cwd() / "config.ini")

OLLAMA_BASE_URL = config.get("ollama", "base_url", fallback="http://localhost:11434/v1")
OLLAMA_MODEL = config.get("ollama", "modelo", fallback="qwen2.5:7b-instruct-q4_K_M")
OLLAMA_TEMPERATURE = config.getfloat("ollama", "temperature", fallback=0.1)
CSV_PATH = config.get("datos", "csv_path", fallback="customer_support_tickets.csv")
MAX_TICKETS_AL_LLM = config.getint("datos", "max_tickets_al_llm", fallback=10)

print(f"Ollama base_url : {OLLAMA_BASE_URL}")
print(f"Modelo          : {OLLAMA_MODEL}")
print(f"CSV             : {CSV_PATH}")

### Instalacion (correr una vez, luego reiniciar el kernel)

In [ ]:
#!pip install -r requirements.txt

### Librerias

In [ ]:
import json
import pandas as pd
import nest_asyncio
from pydantic import BaseModel, Field, field_validator, ValidationError
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

nest_asyncio.apply()

### Dataset

Se lee el CSV local del proyecto (a diferencia del original, que lo bajaba de una URL
publica de IBM). Mantener los datos de clientes dentro de la carpeta del proyecto es
parte de la soberania del dato: nada sale a un bucket externo.

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head(5)

### Esquema de salida (`SupportResponse`)

Igual que en el original: fuerza que el LLM devuelva siempre una estructura valida
(categoria, prioridad, escalamiento, respuesta sugerida), sin importar que tan libre
sea el texto que genera el modelo local.

In [ ]:
class SupportResponse(BaseModel):
    category: str = Field(..., description="The issue category")
    priority: str = Field(default="low", description="Issue priority (low, medium, high)")
    escalate: bool = Field(..., description="Should this be escalated to a human?")
    suggested_response: str = Field(..., description="A helpful support reply")

    @field_validator("priority", mode="before")
    @classmethod
    def validate_priority(cls, v):
        valid_priorities = {"low", "medium", "high"}
        if isinstance(v, str) and v.lower() in valid_priorities:
            return v.lower()
        return "low"

    def priority_as_int(self) -> int:
        priority_map = {"low": 1, "medium": 2, "high": 3}
        return priority_map.get(self.priority, 1)

### Busqueda de tickets en el CSV

El original llamaba a `search_tickets(user_input, df)` sin definirla nunca. Aqui se
implementa: busca coincidencias por Ticket ID, email o nombre de cliente (case-insensitive,
substring), para no depender de que el usuario escriba el dato exacto.

In [ ]:
def search_tickets(user_input: str, data: pd.DataFrame) -> pd.DataFrame:
    texto = user_input.strip().lower()
    if not texto:
        return data.iloc[0:0]

    columnas_email = "Customer Email"
    columnas_nombre = "Customer Name"
    columnas_id = "Ticket ID"

    mask_nombre = data[columnas_nombre].str.lower().str.contains(texto, na=False)
    mask_email = data[columnas_email].str.lower().str.contains(texto, na=False)

    # Ticket ID es numerico: solo intenta calzar si el texto (o un token de el) es un numero
    mask_id = pd.Series(False, index=data.index)
    for token in texto.replace(",", " ").split():
        if token.isdigit():
            mask_id = mask_id | (data[columnas_id] == int(token))

    return data[mask_nombre | mask_email | mask_id]

### Agente PydanticAI apuntando a Ollama local

`OllamaProvider` es el provider dedicado que trae `pydantic-ai` para Ollama (no requiere
api_key ni placeholders). El modelo se sigue construyendo con `OpenAIChatModel` porque
Ollama expone un servidor compatible con el protocolo de chat de OpenAI - es solo el
protocolo de red, nunca sale a internet: todo el trafico va a `localhost:11434`.

In [ ]:
ollama_model = OpenAIChatModel(
    model_name=OLLAMA_MODEL,
    provider=OllamaProvider(base_url=OLLAMA_BASE_URL),
)

agent = Agent(
    ollama_model,
    output_type=SupportResponse,
    model_settings={"temperature": OLLAMA_TEMPERATURE},
    instructions=(
        "You are a helpful customer support assistant.\n"
        "You will receive a customer request and, when available, matching rows from "
        "the customer support ticket database in JSON.\n"
        "Rules:\n"
        "- Respond ONLY using the data provided in the matching ticket JSON, if any.\n"
        "- If no matching ticket data is given, respond politely asking the user to "
        "check their ticket ID, email, or name.\n"
        "- Output MUST follow the SupportResponse structure exactly."
    ),
)

### Prueba no interactiva

Util para validar que el agente + Ollama responden correctamente sin necesitar `input()`
(por ejemplo al ejecutar el notebook en batch).

In [ ]:
async def probar_agente(mensaje: str):
    matches = search_tickets(mensaje, df).head(MAX_TICKETS_AL_LLM)

    if matches.empty:
        prompt = f"Customer request:\n\n{mensaje}\n\nNo matching ticket data was found."
    else:
        ticket_data = matches.to_json(orient="records", indent=2)
        prompt = (
            f"Customer request:\n\n{mensaje}\n\n"
            f"Matching ticket data:\n\n{ticket_data}\n\n"
            "Using ONLY the data above, generate the SupportResponse."
        )

    result = await agent.run(prompt)
    return result.output

respuesta = await probar_agente("Christopher Robbins purchased a Dell XPS. Any updates?")
respuesta

### Chat interactivo (igual espiritu que el notebook original, corregido)

In [ ]:
async def run_chatbot():
    print("Customer Support Chatbot (Ollama local) — escribe 'exit' para salir\n")

    while True:
        user_input = input(" You: ").strip()
        if user_input.lower() == "exit":
            print("Exiting chatbot.")
            break
        if not user_input:
            continue

        try:
            matches = search_tickets(user_input, df)

            if matches.empty:
                print("\nAI Response:")
                print("No matching ticket was found. Please check the ticket ID, email, or customer name.\n")
                continue

            matches = matches.head(MAX_TICKETS_AL_LLM)
            ticket_data = matches.to_json(orient="records", indent=2)
            prompt = (
                f"Customer request:\n\n{user_input}\n\n"
                f"Matching ticket data:\n\n{ticket_data}\n\n"
                "Using ONLY the data above, generate the SupportResponse."
            )

            result = await agent.run(prompt)
            response = result.output

            print("\nAI Response:")
            print(f"Category        : {response.category}")
            print(f"Priority        : {response.priority}")
            print(f"Escalate        : {'Yes' if response.escalate else 'No'}")
            print(f"Suggested Reply : {response.suggested_response}\n")

        except (ValidationError, ValueError, json.JSONDecodeError) as e:
            print(f"\n Could not parse structured output from agent: {type(e).__name__}: {e}\n")
        except Exception as e:
            print(f"\n Error: {type(e).__name__}: {e}\n")

# Ejecutar para lanzar el chat
await run_chatbot()

## Preguntas de prueba sugeridas

- `Christopher Robbins purchased a Dell XPS. If you need more details, please let us know.`
- `My Dell XPS keeps dropping from Wi-Fi every few minutes.`
- `Ticket 4`
